# Práctica guiada: Replicación y Sharding en MongoDB con Docker

## Objetivos

Al finalizar esta práctica serás capaz de:

- desplegar varios contenedores MongoDB con Docker
- configurar un **replica set**
- identificar **primary** y **secondary**
- comprobar que los datos se replican entre nodos
- simular la caída del nodo principal
- desplegar un **cluster shardeado**
- comprobar cómo MongoDB reparte datos entre shards
- diferenciar claramente **replicación** y **sharding**

---

## Idea clave

**Replica set**:
- los datos se **replican**
- hay un **primary** y uno o varios **secondary**
- sirve para **alta disponibilidad**

**Sharding**:
- los datos se **distribuyen**
- no todos los documentos están en todos los nodos
- sirve para **escalado horizontal**

# Requisitos previos

Antes de empezar, comprueba que tienes:

- Docker instalado
- Docker Compose disponible
- Puertos libres para los contenedores
- Terminal o consola

Puedes comprobar Docker con:

```bash
docker --version
docker compose version
```

In [ ]:
# Este cuaderno es una guía didáctica.
# La mayoría de comandos deben ejecutarse en la TERMINAL del sistema,
# no dentro del kernel Python del notebook.
#
# Puedes usar estas celdas como referencia, copiar y pegar en la terminal,
# o adaptarlas según tu sistema.

# Parte A — Replica Set

En esta parte vas a crear 3 contenedores MongoDB:

- `mongo1`
- `mongo2`
- `mongo3`

Los tres formarán un **replica set** llamado `rs0`.

## Qué deben observar los alumnos

- qué nodo es **PRIMARY**
- qué nodos son **SECONDARY**
- que un documento insertado en el primary aparece también en los secundarios
- qué pasa si se apaga el primary

## 1. Crear el archivo `docker-compose.yml`

Guarda el siguiente contenido en un archivo llamado `docker-compose.yml`.

In [ ]:
docker_compose_replica = r"""
version: "3.9"

services:
  mongo1:
    image: mongo:latest
    container_name: mongo1
    command: ["mongod", "--replSet", "rs0", "--bind_ip_all", "--port", "27017"]
    ports:
      - "27017:27017"
    volumes:
      - mongo1_data:/data/db
    networks:
      - mongo_cluster

  mongo2:
    image: mongo:latest
    container_name: mongo2
    command: ["mongod", "--replSet", "rs0", "--bind_ip_all", "--port", "27017"]
    ports:
      - "27018:27017"
    volumes:
      - mongo2_data:/data/db
    networks:
      - mongo_cluster

  mongo3:
    image: mongo:latest
    container_name: mongo3
    command: ["mongod", "--replSet", "rs0", "--bind_ip_all", "--port", "27017"]
    ports:
      - "27019:27017"
    volumes:
      - mongo3_data:/data/db
    networks:
      - mongo_cluster

volumes:
  mongo1_data:
  mongo2_data:
  mongo3_data:

networks:
  mongo_cluster:
"""
print(docker_compose_replica)

## 2. Levantar los contenedores

Ejecuta en la terminal:

In [ ]:
# TERMINAL
# docker compose up -d
# docker ps

## 3. Inicializar el replica set

Entra en `mongo1`:

In [ ]:
# TERMINAL
# docker exec -it mongo1 mongosh

Dentro de `mongosh`, ejecuta:

In [ ]:
# EN MONGOSH
rs_config = r"""
rs.initiate({
  _id: "rs0",
  members: [
    { _id: 0, host: "mongo1:27017" },
    { _id: 1, host: "mongo2:27017" },
    { _id: 2, host: "mongo3:27017" }
  ]
})
"""
print(rs_config)

Después comprueba el estado:

In [ ]:
# EN MONGOSH
# rs.status()
# db.hello()

## 4. Insertar datos en el primary

Ejemplo:

In [ ]:
# EN MONGOSH
replica_insert = r"""
use tienda

db.productos.insertMany([
  { nombre: "Portátil", precio: 900, stock: 10 },
  { nombre: "Teclado", precio: 45, stock: 30 },
  { nombre: "Ratón", precio: 20, stock: 50 }
])

db.productos.find()
"""
print(replica_insert)

## 5. Leer desde un secondary

Abre otra terminal y entra en `mongo2`:

In [ ]:
# TERMINAL
# docker exec -it mongo2 mongosh

Permite lectura en secundario:

In [ ]:
# EN MONGOSH
secondary_read = r"""
db.getMongo().setReadPref("secondaryPreferred")
use tienda
db.productos.find()
"""
print(secondary_read)

Repite también en `mongo3`.

## Qué deben observar

Los documentos insertados en el primary aparecen también en los secundarios.  
Eso significa que aquí los datos **se replican**.

## 6. Simular la caída del primary

Primero identifica cuál es el nodo primary.  
Después, desde otra terminal:

In [ ]:
# TERMINAL
# docker stop mongo1

Espera unos segundos y vuelve a comprobar el estado desde `mongo2` o `mongo3`:

In [ ]:
# EN MONGOSH
# rs.status()
# db.hello()

## Qué deben observar

- otro nodo pasa a ser **PRIMARY**
- el sistema sigue funcionando

Eso demuestra la utilidad del replica set para alta disponibilidad.

## 7. Levantar de nuevo el nodo detenido

In [ ]:
# TERMINAL
# docker start mongo1

Comprueba después otra vez:

```javascript
rs.status()
```

# Actividad de reflexión — Replica Set

Responde:

1. ¿Los datos se reparten o se copian entre nodos?
2. ¿Qué nodo acepta escrituras?
3. ¿Qué ocurre cuando cae el primary?
4. ¿Para qué usarías un replica set en un sistema real?

# Parte B — Sharding

En esta parte vas a montar un entorno de sharding con:

- 1 **config server**
- 2 **shards**
- 1 **mongos router**

## Qué deben observar los alumnos

- que los datos ya no están todos en los mismos nodos
- que MongoDB reparte documentos entre shards
- que eso depende de una **shard key**

## 1. Crear el archivo `docker-compose-sharding.yml`

In [ ]:
docker_compose_sharding = r"""
version: "3.9"

services:
  configsvr:
    image: mongo:latest
    container_name: configsvr
    command: mongod --configsvr --replSet configReplSet --bind_ip_all --port 27017
    ports:
      - "27100:27017"
    volumes:
      - config_data:/data/db
    networks:
      - mongo_shard_net

  shard1:
    image: mongo:latest
    container_name: shard1
    command: mongod --shardsvr --replSet shard1ReplSet --bind_ip_all --port 27017
    ports:
      - "27101:27017"
    volumes:
      - shard1_data:/data/db
    networks:
      - mongo_shard_net

  shard2:
    image: mongo:latest
    container_name: shard2
    command: mongod --shardsvr --replSet shard2ReplSet --bind_ip_all --port 27017
    ports:
      - "27102:27017"
    volumes:
      - shard2_data:/data/db
    networks:
      - mongo_shard_net

  mongos:
    image: mongo:latest
    container_name: mongos
    depends_on:
      - configsvr
      - shard1
      - shard2
    command: mongos --configdb configReplSet/configsvr:27017 --bind_ip_all --port 27017
    ports:
      - "27020:27017"
    networks:
      - mongo_shard_net

volumes:
  config_data:
  shard1_data:
  shard2_data:

networks:
  mongo_shard_net:
"""
print(docker_compose_sharding)

## 2. Levantar el entorno shardeado

In [ ]:
# TERMINAL
# docker compose -f docker-compose-sharding.yml up -d
# docker ps

## 3. Inicializar el config server

In [ ]:
# TERMINAL
# docker exec -it configsvr mongosh

In [ ]:
# EN MONGOSH
config_init = r"""
rs.initiate({
  _id: "configReplSet",
  configsvr: true,
  members: [
    { _id: 0, host: "configsvr:27017" }
  ]
})
"""
print(config_init)

## 4. Inicializar el shard 1

In [ ]:
# TERMINAL
# docker exec -it shard1 mongosh

In [ ]:
# EN MONGOSH
shard1_init = r"""
rs.initiate({
  _id: "shard1ReplSet",
  members: [
    { _id: 0, host: "shard1:27017" }
  ]
})
"""
print(shard1_init)

## 5. Inicializar el shard 2

In [ ]:
# TERMINAL
# docker exec -it shard2 mongosh

In [ ]:
# EN MONGOSH
shard2_init = r"""
rs.initiate({
  _id: "shard2ReplSet",
  members: [
    { _id: 0, host: "shard2:27017" }
  ]
})
"""
print(shard2_init)

## 6. Conectar al router `mongos`

In [ ]:
# TERMINAL
# docker exec -it mongos mongosh

Añadir los shards:

In [ ]:
# EN MONGOSH
mongos_add_shards = r"""
sh.addShard("shard1ReplSet/shard1:27017")
sh.addShard("shard2ReplSet/shard2:27017")
sh.status()
"""
print(mongos_add_shards)

## 7. Habilitar sharding para una base de datos

In [ ]:
# EN MONGOSH
enable_sharding = r"""
sh.enableSharding("instituto")
use instituto
db.alumnos.createIndex({ brigada: 1 })
sh.shardCollection("instituto.alumnos", { brigada: 1 })
"""
print(enable_sharding)

## 8. Insertar datos

In [ ]:
# EN MONGOSH
insert_sharded_data = r"""
for (let i = 1; i <= 50; i++) {
  db.alumnos.insertOne({
    brigada: i,
    nombre: "Alumno" + i,
    nota: Math.floor(Math.random() * 10) + 1
  })
}
"""
print(insert_sharded_data)

## 9. Ver distribución de datos

In [ ]:
# EN MONGOSH
# db.alumnos.getShardDistribution()
# sh.status()

## 10. Consultar directamente cada shard

En `shard1`:

In [ ]:
# TERMINAL
# docker exec -it shard1 mongosh
# use instituto
# db.alumnos.find()

En `shard2`:

In [ ]:
# TERMINAL
# docker exec -it shard2 mongosh
# use instituto
# db.alumnos.find()

## Qué deben observar

Ahora los datos **no están todos en ambos nodos**.  
Aquí los documentos se **reparten** entre los shards.

Eso significa que **sharding no es lo mismo que replicación**.

# Actividades finales

## Actividad A — Replica set

1. Levanta el replica set.
2. Identifica primary y secondary.
3. Inserta 5 documentos en `tienda.productos`.
4. Comprueba que aparecen en los otros nodos.
5. Apaga el primary.
6. Observa qué nodo se convierte en primary.
7. Vuelve a arrancar el nodo detenido.

## Actividad B — Sharding

1. Levanta el cluster shardeado.
2. Añade los shards.
3. Activa sharding en la base `instituto`.
4. Crea la colección `alumnos`.
5. Usa `brigada` como shard key.
6. Inserta 50 documentos.
7. Consulta la distribución con `getShardDistribution()`.
8. Comprueba manualmente qué documentos hay en cada shard.

# Preguntas de reflexión

1. ¿Qué diferencia hay entre **replicar** y **fragmentar** datos?
2. ¿En qué arquitectura se copia la información completa?
3. ¿En cuál se reparten los documentos?
4. ¿Qué ventajas aporta un replica set?
5. ¿Qué ventajas aporta el sharding?
6. ¿Cuál de los dos montajes te parece más complejo y por qué?

# Conclusión

- **Replica set** = alta disponibilidad y tolerancia a fallos
- **Sharding** = escalado horizontal y reparto de carga
- En sistemas reales, MongoDB puede combinar ambas estrategias